## Learning about cache() and persist()

> **在 Spark 的默认宇宙里，所有的 DataFrame 都是“渣男”——它们是【没有记忆】的。**
> **当你写下 `df2 = df1.filter(...)`，然后对 `df2` 执行 `show()` 时，Spark 会从头读源表、过滤、输出。如果你接着对 `df2` 执行 `count()`，Spark 绝对不会复用刚才的结果，而是【死板地再次从头读一遍源表、再次过滤一遍、再次计算 count】。**
> **如果你的上游是一个经过了 30 次复杂 Join、耗时 1 小时的巨型管道，下游有 5 个报表都要用这个结果，这就意味着你要把这 1 小时的地狱计算死板地【重复跑 5 遍】！网线和 CPU 会被活生生烧穿。**
> **`cache()` 和 `persist()` 就是为了打破这个死板循环而生的【时空定格锚点】。它让数据在流经某一个关键节点时，原地“肉身飞升”，把计算好的明细直接锁死在内存里。下游再来要数据？别算老本了，直接去内存里拿现成的！**

---

### 一、 cache() 与 persist() 的底层物理原理

在底层，它们是**母子关系**。

#### 1. 物理本质

* **`cache()`**：是一个快捷包装。它的底层就是调用了 `persist()`，并且死板地指定了缓存级别为 **`MEMORY_AND_DISK`（内存不够写磁盘）**。
* **`persist(level)`**：是真正掌管时空结界的底层大将。它允许架构师**手动挑选**数据在物理硬件上的存放姿态。

#### 2. 工业界最常用的 3 大缓存级别（Storage Level）

在重工业生产中，你必须根据机房的硬件水位，精准挑选以下三个级别之一（需要先导入 `from pyspark import StorageLevel`）：

* **`MEMORY_ONLY`**：
* *大白话*：纯内存。如果内存装得下就装，装不下（OOM 边缘）的碎片就**直接丢弃**！当下游用到那些丢失的分区时，系统会无奈地重新从头算一遍。


* **`MEMORY_AND_DISK`（`cache()` 的默认行为）**：
* *大白话*：内存 + 磁盘备胎。内存塞满了，剩下的数据自动无痛写进本地机器的机械硬盘/SSD里。**最安全，但如果大量写盘会触发磁盘 I/O 变慢。**


* **`MEMORY_ONLY_SER` / `MEMORY_AND_DISK_SER`（大厂高级调优必选）**：
* *大白话*：**序列化纯内存/序列化备胎**。把数据在内存里压缩成一条条冰冷的字节数组（Byte Array）。
* *架构师账本*：它能让原本占 10GB 的对象在内存里瞬间暴跌到 2GB，**极大地拯救了内存空间**！唯一的代价是下游读取时，CPU 需要额外花一丁点算力去“解压（反序列化）”。



---

### 二、 什么时候才“真正需要”缓存？（合适场景）

重工业铁律：**缓存是用来“切断重复计算”的，不是用来“加速单次查询”的！**

只有当你的代码同时满足以下两个红线，才允许写下 `.cache()`：

1. **一源多用（The Multiple Actions Rule）**：
同一个 DataFrame，在后续的代码里被用到了 **2 次及以上**（比如既要写入数据库，又要算 count，还要发 Kafka）。
2. **上游代价极高（High-Cost Lineage）**：
这个 DataFrame 上游经历了极其痛苦的 `Shuffle`（如大表 Join、多层聚合、或复杂的正则清洗）。把它锁在内存里，能瞬间免除下游无数次重复洗牌的灾难。

---

### 三、 斩断毒瘤：避免滥用缓存

在初学者眼里，`.cache()` 像是兴奋剂，恨不得每写三行代码就加一个。这在工业界是**极其致命的毒瘤行为**：

#### 1. 为什么滥用缓存会反噬系统？

* **内存窒息（Cache Eviction Storm）**：Spark 的 Executor 内存是共享的。执行计算需要“执行内存（Execution Memory）”，缓存占用“存储内存（Storage Memory）”。
* 如果你把不用的垃圾表也长期锁在内存里，执行内存就会被疯狂压缩。系统为了维持运转，会频繁触发垃圾回收（Full GC）或者把缓存疯狂往磁盘上倒腾（Disk Spilling），**反而让原本 1 秒跑完的任务生生拖到 1 小时卡死**。

#### 2. 架构师的解药：用完必须斩立决（`unpersist`）

缓存在完成了它的历史使命后，必须在代码里显式地释放掉，让内存重新呼吸！

```python
# 1. 锁死内存
df_critical.cache()

# 2. 下游疯狂复用
df_critical.write.saveAsTable("table_1")
count_res = df_critical.count()

# 3. 🚀 使命完成，斩立决！立刻把内存吐出来还给集群
df_critical.unpersist()

```



做AB实验，模拟一个代价高昂的上游。A不用cache，B用cache，对比两者。

In [0]:
import time

# 1. 模拟一个高昂代价的上游200万行数据。 并且为了区分强制进行一次Shuffle洗牌打散

# 用repartition就是强行分区，增加负担
df_heavy = spark.range(0,2000000).repartition(10)  

start_1 = time.time()
count_1 = df_heavy.count()
end_1 = time.time()
print(f"The first time: {end_1-start_1}")

start_2 = time.time()
count_2 = df_heavy.count()
end_2 = time.time()
print(f"The second time: {end_2-start_2}")


可以发现，其实第二次和第一次运行时间运行差异不大，证明第二次运行也是重新计算，并没有复用第一次的结果，
所以我们做一个B实验，唯一的区别就是加了缓存cache()

In [0]:
# B experiment: use cache()
df_heavy_cache = spark.range(0,2000000).repartition(10)  

df_heavy_cache.cache()

start_1 = time.time()
count_1 = df_heavy_cache.count()
end_1 = time.time()
print(f"time: {end_1-start_1}")

start_2 = time.time()
count_2 = df_heavy_cache.count()
end_2 = time.time()
print(f"time: {end_2-start_2}")

df_heavy_cache.unpersist()

In [0]:
import time

# 1. 模拟地狱级痛苦的上游管道（200万行，强制洗牌）
df_heavy_source = spark.range(0, 2000000).repartition(10)

# 2. 🚀 【第一聚合期】：将结果直接落盘成临时 Delta 表。
# 这一步，不仅完成了 Shuffle，更重要的是：Delta 引擎会顺手在元数据里把统计结果死死锁住！
df_heavy_source.write.format("delta").mode("overwrite").saveAsTable("yuto_heavy_checkpoint")


# ==========================================
# 🛰️ 🚀 见证奇迹：下游多路大军，直接读取这个干净的“中转站”表
# ==========================================
df_checkpoint = spark.table("yuto_heavy_checkpoint")

# 📊 第一次调用：看看要多久
start_a = time.time()
count_a = df_checkpoint.count()
print(f"⏱️ 报表 A（第一次读中转表）耗时: {time.time() - start_a:.4f} 秒")

# 📊 第二次调用：再次计算 count()
start_b = time.time()
count_b = df_checkpoint.count()
print(f"🎉 报表 B（第二次无痛秒杀）耗时: {time.time() - start_b:.4f} 秒")


# 3. 洗刷战场
spark.sql("DROP TABLE IF EXISTS yuto_heavy_checkpoint")